#  Egyptian License Plate Recognition - Character Classifier V2
### Project: EALPR Professional Training Pipeline

This notebook implements an optimized training pipeline with a validation split to ensure high generalization accuracy.

**Core Features:**
1. **Train/Val Split (80/20)**: Real-time accuracy monitoring on unseen data.
2. **EfficientNet-B0**: State-of-the-art CNN architecture.
3. **Heavy-Duty Augmentation**: Specialized for low-quality plate images.
4. **Best Model Saving**: Automatically saves the version with the highest validation accuracy.

In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, WeightedRandomSampler, Subset
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageFilter, ImageOps
from tqdm.auto import tqdm

# Check for GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on: {device}")

Training on: cpu


## 1. Custom Preprocessing
Ensuring characters are padded to square shapes to prevent distortion during resizing.

In [2]:
class HeavyDutyTransform:
    def __call__(self, img):
        # 1. Padding to Square (Maintain aspect ratio)
        w, h = img.size
        side = max(w, h)
        pad_img = Image.new("L", (side, side), 255) # White background
        pad_img.paste(img, ((side - w) // 2, (side - h) // 2))
        
        # 2. Resize to model input size and Sharpen
        img = pad_img.resize((128, 128), Image.BILINEAR)
        img = img.filter(ImageFilter.SHARPEN)
        img = ImageOps.autocontrast(img)
        
        return img.convert("RGB")

## 2. Dataset Setup & Validation Split

In [3]:
DATA_DIR = 'Characters'

train_transforms = transforms.Compose([
    HeavyDutyTransform(),
    transforms.RandomRotation(15),
    transforms.RandomAffine(0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    HeavyDutyTransform(),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Load full dataset
full_dataset = datasets.ImageFolder(DATA_DIR)
class_names = full_dataset.classes
print(f"Found {len(class_names)} classes.")

# Calculate sizes for 80/10/10 split
total_size = len(full_dataset)
train_size = int(0.8 * total_size)
val_size = int(0.1 * total_size)
test_size = total_size - train_size - val_size

train_subset, val_subset, test_subset = torch.utils.data.random_split(
    full_dataset, [train_size, val_size, test_size]
)

# Assign transforms
train_subset.dataset.transform = train_transforms
val_subset.dataset.transform = val_transforms
test_subset.dataset.transform = val_transforms

# Sampler for training set imbalance
train_indices = train_subset.indices
train_targets = [full_dataset.targets[i] for i in train_indices]
class_sample_count = np.array([len(np.where(np.array(train_targets) == t)[0]) for t in np.unique(train_targets)])
weight = 1. / class_sample_count
samples_weight = torch.from_numpy(weight[np.array(train_targets)]).double()
sampler = WeightedRandomSampler(samples_weight, len(samples_weight))

train_loader = DataLoader(train_subset, batch_size=32, sampler=sampler)
val_loader = DataLoader(val_subset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_subset, batch_size=32, shuffle=False)

print(f"Split Complete: {train_size} Train | {val_size} Val | {test_size} Test")

Found 26 classes.
Split Complete: 11124 Train | 1390 Val | 1392 Test


## 3. Build EfficientNet-B0 Model
Loading pre-trained weights and replacing the head for our specific classes.

In [4]:
def build_model(num_classes):
    # Loads fresh ImageNet weights to clear previous training session
    model = models.efficientnet_b0(weights='IMAGENET1K_V1')
    in_features = model.classifier[1].in_features
    model.classifier[1] = nn.Sequential(
        nn.Linear(in_features, 512),
        nn.ReLU(),
        nn.Dropout(0.3),
        nn.Linear(512, num_classes)
    )
    return model

model = build_model(len(class_names)).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

## 4. Training and Evaluation Loop

In [5]:
epochs = 20
best_val_acc = 0

for epoch in range(epochs):
    # --- TRAINING ---
    model.train()
    train_correct = 0
    train_loop = tqdm(train_loader, leave=False)
    for imgs, labels in train_loop:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        out = model(imgs)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        _, pred = out.max(1)
        train_correct += pred.eq(labels).sum().item()
        train_loop.set_description(f"Epoch [{epoch+1}/{epochs}]")
        train_loop.set_postfix(acc=f"{(100*train_correct/((train_loop.n+1)*32)):.2f}%")

    # --- VALIDATION ---
    model.eval()
    val_correct = 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            out = model(imgs)
            _, pred = out.max(1)
            val_correct += pred.eq(labels).sum().item()
    
    val_acc = 100 * val_correct / len(val_subset)
    print(f"Epoch [{epoch+1}/{epochs}] - Val Acc: {val_acc:.2f}%")
    
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'models/efficientnet_b0_master.pth')
        print(f" --> Saved Best Model!")

# --- FINAL TESTING (Unseen by Val or Train) ---
print("\n--- Final Test Evaluation ---")
model.load_state_dict(torch.load('models/efficientnet_b0_master.pth'))
model.eval()
test_correct = 0
with torch.no_grad():
    for imgs, labels in test_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        out = model(imgs)
        _, pred = out.max(1)
        test_correct += pred.eq(labels).sum().item()

final_test_acc = 100 * test_correct / len(test_subset)
print(f"FINAL PROJECT ACCURACY (TEST SET): {final_test_acc:.2f}%")

  0%|          | 0/348 [00:00<?, ?it/s]

Epoch [1/20] - Val Acc: 96.91%
 --> Saved Best Model!


  0%|          | 0/348 [00:00<?, ?it/s]

Epoch [2/20] - Val Acc: 99.71%
 --> Saved Best Model!


  0%|          | 0/348 [00:00<?, ?it/s]

...

Epoch [18/20] - Val Acc: 99.86%


  0%|          | 0/348 [00:00<?, ?it/s]

Epoch [19/20] - Val Acc: 99.86%


  0%|          | 0/348 [00:00<?, ?it/s]

Epoch [20/20] - Val Acc: 99.78%

--- Final Test Evaluation ---
FINAL PROJECT ACCURACY (TEST SET): 99.86%
